# IRIS: World Model as a Language Model — From Scratch

**Paper:** [Transformers are Sample-Efficient World Models](https://arxiv.org/abs/2209.00588) (ICLR 2023)

---

### The Big Idea

What if we treated world modeling exactly like language modeling?

- **Language model:** Tokenize text → predict next token with a Transformer
- **IRIS:** Tokenize *images* into discrete tokens → predict next *visual* tokens (+ reward + done) with a Transformer

Then, instead of interacting with the real environment, we train a policy **entirely inside the world model's dreams** — the Transformer imagines future frames, and the agent learns from those imagined experiences.

### Architecture (3 Components)

```
1. TOKENIZER (VQ-VAE)
   64×64 RGB image → Encoder → Vector Quantize → 16 discrete tokens (from codebook of 256)
   16 discrete tokens → Decoder → 64×64 RGB reconstruction

2. WORLD MODEL (GPT-like Transformer)
   [obs₀ obs₁ ... obs₁₅ act] × T steps → predict next obs tokens + reward + done
   (Exactly like GPT predicting next words, but over visual tokens + actions)

3. ACTOR-CRITIC (CNN + LSTM)
   Reconstructed image → CNN → LSTM → action logits + value
   Trained entirely in the world model's "dreams" (imagined rollouts)
```

### vs. DINO-WM
| | DINO-WM | IRIS |
|---|---|---|
| **Representation** | Continuous (DINO features) | Discrete (VQ tokens) |
| **Encoder** | Frozen pre-trained DINO | Trained VQ-VAE |
| **Prediction** | MSE in feature space | Cross-entropy over token vocabulary |
| **Planning** | CEM in embedding space | Train a full RL policy in dreams |
| **Key insight** | Pre-trained features are powerful | Discretization enables GPT-style modeling |

### Environment: Catch
We'll use a simple **Catch** game: a ball drops from the top, and a paddle at the bottom moves left/right to catch it. Simple enough to train in ~15 min on T4, complex enough to need all three components.

**Runtime:** ~15 min on T4 GPU

In [ ]:
!pip install -q matplotlib tqdm scikit-learn

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('medium')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## 1. The Catch Environment

A ball drops from a random column at the top. The paddle (3 pixels wide) sits at the bottom row. The agent can move **left**, **stay**, or **right**.

- **Reward:** +1 if the ball lands on the paddle, −1 if it misses
- **Observation:** 64×64 RGB image (ball = red, paddle = green, background = dark)
- **Actions:** 0=left, 1=stay, 2=right
- **Episode length:** ball takes ~15 steps to fall

In [ ]:
class CatchEnv:
    """Simple Catch game: ball falls, paddle catches.

    DENSE REWARDS (critical for world model dream training):
      - Every step: small reward based on paddle-ball alignment
      - Terminal: +1 (catch) or -1 (miss)

    Without dense rewards, the world model learns to always predict reward=0
    and the actor-critic gets no learning signal in dreams.
    """

    def __init__(self, size=64, paddle_width=5, ball_speed=4):
        self.size = size
        self.pw = paddle_width
        self.ball_speed = ball_speed
        self.num_actions = 3  # left, stay, right
        self.reset()

    def reset(self):
        self.ball_x = np.random.randint(4, self.size - 4)
        self.ball_y = 2
        self.paddle_x = self.size // 2
        self.done = False
        return self._render()

    def step(self, action):
        # Move paddle: 0=left, 1=stay, 2=right
        move = (action - 1) * 4  # -4, 0, +4
        self.paddle_x = np.clip(self.paddle_x + move, self.pw, self.size - self.pw - 1)

        # Move ball down
        self.ball_y += self.ball_speed

        self.done = False

        # Dense shaping reward: how close is the paddle to being under the ball?
        distance = abs(self.ball_x - self.paddle_x) / self.size  # [0, 1]
        reward = 0.1 * (1.0 - 2.0 * distance)  # +0.1 when aligned, -0.1 when far

        # Check if ball reached bottom
        if self.ball_y >= self.size - 6:
            self.done = True
            if abs(self.ball_x - self.paddle_x) <= self.pw + 1:
                reward = 1.0   # caught!
            else:
                reward = -1.0  # missed!

        return self._render(), reward, self.done

    def _render(self):
        """Render 64x64 RGB image (uint8)."""
        img = np.full((self.size, self.size, 3), 20, dtype=np.uint8)  # dark bg

        # Draw ball (red circle, radius 3)
        by, bx = int(self.ball_y), int(self.ball_x)
        for dy in range(-3, 4):
            for dx in range(-3, 4):
                if dy*dy + dx*dx <= 9:
                    yy, xx = by + dy, bx + dx
                    if 0 <= yy < self.size and 0 <= xx < self.size:
                        img[yy, xx] = [220, 60, 60]

        # Draw paddle (green rectangle)
        py = self.size - 4
        px = int(self.paddle_x)
        for dx in range(-self.pw, self.pw + 1):
            for dy in range(-1, 2):
                xx = px + dx
                yy = py + dy
                if 0 <= xx < self.size and 0 <= yy < self.size:
                    img[yy, xx] = [60, 200, 80]

        return img

# Test the environment
env = CatchEnv()
fig, axes = plt.subplots(1, 6, figsize=(15, 2.5))
obs = env.reset()
total_r = 0
for i in range(6):
    axes[i].imshow(obs); axes[i].axis('off')
    obs, r, d = env.step(1)  # stay
    total_r += r
    axes[i].set_title(f'r={r:.2f}', fontsize=9)
    if d: break
fig.suptitle('Catch Environment — dense reward every step (green = aligned, red = far)', fontsize=11)
plt.tight_layout(); plt.show()
print(f"Dense rewards give the world model something to predict EVERY step, not just the last.")

---
## 2. Collect Experience

Before training any component, we collect a buffer of episodes by playing randomly. This gives us data for training the tokenizer and world model.

In [ ]:
class ReplayBuffer:
    """Stores complete episodes of (observations, actions, rewards, dones)."""

    def __init__(self):
        self.episodes = []  # list of dicts

    def add_episode(self, obs_list, act_list, rew_list, done_list):
        self.episodes.append({
            'observations': np.stack(obs_list),   # (T+1, 64, 64, 3) uint8 — includes reset frame
            'actions': np.array(act_list),         # (T,) int
            'rewards': np.array(rew_list),         # (T,) float
            'dones': np.array(done_list),          # (T,) bool
        })

    def sample_frames(self, batch_size):
        """Sample random individual frames (for tokenizer training)."""
        frames = []
        for _ in range(batch_size):
            ep = self.episodes[np.random.randint(len(self.episodes))]
            t = np.random.randint(len(ep['observations']))
            frames.append(ep['observations'][t])
        imgs = np.stack(frames)  # (B, 64, 64, 3) uint8
        return torch.from_numpy(imgs).permute(0, 3, 1, 2).float() / 255.0  # (B, 3, 64, 64)

    def sample_sequences(self, batch_size, seq_len):
        """Sample random contiguous sequences (for world model training).
        Uses actions length (T) as the reference — obs has T+1 frames."""
        obs_b, act_b, rew_b, done_b = [], [], [], []
        for _ in range(batch_size):
            # Pick episode with enough actions
            while True:
                ep = self.episodes[np.random.randint(len(self.episodes))]
                if len(ep['actions']) >= seq_len:
                    break
            t0 = np.random.randint(0, len(ep['actions']) - seq_len + 1)
            # obs[t0:t0+seq_len] aligns with actions[t0:t0+seq_len]
            # (obs[t] is the observation BEFORE taking actions[t])
            obs_b.append(ep['observations'][t0:t0+seq_len])
            act_b.append(ep['actions'][t0:t0+seq_len])
            rew_b.append(ep['rewards'][t0:t0+seq_len])
            done_b.append(ep['dones'][t0:t0+seq_len])
        obs = torch.from_numpy(np.stack(obs_b)).permute(0, 1, 4, 2, 3).float() / 255.0
        act = torch.from_numpy(np.stack(act_b)).long()
        rew = torch.from_numpy(np.stack(rew_b)).float()
        done = torch.from_numpy(np.stack(done_b)).long()
        return obs, act, rew, done  # (B,T,3,64,64), (B,T), (B,T), (B,T)


# Collect episodes with random policy
buffer = ReplayBuffer()
env = CatchEnv()
N_EPISODES = 2000

total_catches = 0
for _ in tqdm(range(N_EPISODES), desc='Collecting episodes'):
    obs = env.reset()
    obs_list, act_list, rew_list, done_list = [obs], [], [], []
    done = False
    while not done:
        action = np.random.randint(3)
        obs, reward, done = env.step(action)
        obs_list.append(obs)
        act_list.append(action)
        rew_list.append(reward)
        done_list.append(done)
    if reward > 0:
        total_catches += 1
    buffer.add_episode(obs_list, act_list, rew_list, done_list)

print(f"\nCollected {N_EPISODES} episodes")
print(f"Random policy catch rate: {total_catches/N_EPISODES:.1%}")
print(f"Avg episode length: {np.mean([len(e['actions']) for e in buffer.episodes]):.1f} steps")

---
## 3. Component 1: VQ-VAE Tokenizer

The tokenizer converts a 64×64 RGB image into **16 discrete tokens** (a 4×4 grid), each chosen from a **codebook of 256 entries**.

This is the bridge between continuous pixels and discrete "language" that the Transformer will model.

```
Image (3, 64, 64)
  → Encoder CNN → continuous latent (256, 4, 4)
  → Vector Quantize → 16 discrete tokens in [0, 255]
  → Decoder CNN → reconstructed image (3, 64, 64)
```

**Key concept:** The codebook is like a vocabulary. Each 4×4 patch region of the image gets mapped to the nearest "word" in this visual vocabulary. This is exactly analogous to BPE tokenization in language models.

In [ ]:
# ── Building Blocks ─────────────────────────────────────────────────

class ResBlock(nn.Module):
    """Residual block with GroupNorm + SiLU."""
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.GroupNorm(8, channels),
            nn.SiLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.GroupNorm(8, channels),
            nn.SiLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
        )

    def forward(self, x):
        return x + self.block(x)


class VectorQuantizer(nn.Module):
    """
    Vector Quantization: maps each continuous vector to the nearest codebook entry.

    This is the core of the tokenizer — it creates a discrete bottleneck.
    The straight-through estimator copies gradients from decoder to encoder,
    bypassing the non-differentiable argmin.
    """
    def __init__(self, num_embeddings=256, embedding_dim=256, beta=0.25):
        super().__init__()
        self.K = num_embeddings
        self.D = embedding_dim
        self.beta = beta
        self.codebook = nn.Embedding(num_embeddings, embedding_dim)
        # Initialize codebook uniformly
        self.codebook.weight.data.uniform_(-1.0 / num_embeddings, 1.0 / num_embeddings)

    def forward(self, z):
        """
        z: (B, D, H, W) continuous latent
        Returns: z_q (quantized), tokens (indices), vq_loss
        """
        B, D, H, W = z.shape
        # Reshape to (B*H*W, D) for distance computation
        z_flat = z.permute(0, 2, 3, 1).reshape(-1, D)  # (B*H*W, D)

        # Compute distances to all codebook entries
        # ||z - e||^2 = ||z||^2 + ||e||^2 - 2*z·e
        dist = (z_flat.pow(2).sum(1, keepdim=True)
                + self.codebook.weight.pow(2).sum(1)
                - 2 * z_flat @ self.codebook.weight.t())  # (B*H*W, K)

        # Find nearest codebook entry
        tokens = dist.argmin(dim=-1)  # (B*H*W,)
        z_q = self.codebook(tokens).reshape(B, H, W, D).permute(0, 3, 1, 2)  # (B, D, H, W)

        # VQ losses
        # 1. Commitment loss: push encoder output toward codebook (gradients to encoder)
        # 2. Codebook loss: push codebook toward encoder output (gradients to codebook)
        commitment = F.mse_loss(z, z_q.detach())
        codebook_loss = F.mse_loss(z.detach(), z_q)
        vq_loss = codebook_loss + self.beta * commitment

        # Straight-through estimator: copy gradients from z_q to z
        z_q = z + (z_q - z).detach()

        tokens = tokens.reshape(B, H, W)  # (B, 4, 4)
        return z_q, tokens, vq_loss


print("VQ-VAE building blocks defined.")
print("Key idea: Vector Quantization creates a discrete bottleneck — ")
print("continuous encoder outputs get snapped to the nearest codebook vector.")

In [ ]:
class Tokenizer(nn.Module):
    """
    VQ-VAE Tokenizer: Image → 16 discrete tokens → Reconstructed image.

    Encoder:  (3, 64, 64) → downsample 4x → (256, 4, 4)
    Quantize: (256, 4, 4) → 16 tokens from codebook of 256
    Decoder:  (256, 4, 4) → upsample 4x → (3, 64, 64)
    """
    def __init__(self, embed_dim=256, codebook_size=256):
        super().__init__()
        self.embed_dim = embed_dim
        self.codebook_size = codebook_size

        # Encoder: 64→32→16→8→4
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, 4, stride=2, padding=1),   # 64→32
            nn.SiLU(),
            ResBlock(64),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),  # 32→16
            nn.SiLU(),
            ResBlock(128),
            nn.Conv2d(128, 256, 4, stride=2, padding=1), # 16→8
            nn.SiLU(),
            ResBlock(256),
            nn.Conv2d(256, embed_dim, 4, stride=2, padding=1), # 8→4
            nn.SiLU(),
            ResBlock(embed_dim),
        )

        # Vector Quantizer
        self.vq = VectorQuantizer(codebook_size, embed_dim)

        # Decoder: 4→8→16→32→64
        self.decoder = nn.Sequential(
            ResBlock(embed_dim),
            nn.ConvTranspose2d(embed_dim, 256, 4, stride=2, padding=1), # 4→8
            nn.SiLU(),
            ResBlock(256),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),  # 8→16
            nn.SiLU(),
            ResBlock(128),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),   # 16→32
            nn.SiLU(),
            ResBlock(64),
            nn.ConvTranspose2d(64, 3, 4, stride=2, padding=1),     # 32→64
            nn.Sigmoid(),  # output in [0, 1]
        )

    def encode(self, x):
        """Image (B,3,64,64) → tokens (B,4,4) integers in [0, codebook_size)."""
        z = self.encoder(x)          # (B, 256, 4, 4)
        z_q, tokens, vq_loss = self.vq(z)
        return tokens, z_q, vq_loss

    def decode(self, z_q):
        """Quantized latent (B,256,4,4) → reconstructed image (B,3,64,64)."""
        return self.decoder(z_q)

    def decode_tokens(self, tokens):
        """Token indices (B,4,4) → reconstructed image (B,3,64,64)."""
        B, H, W = tokens.shape
        z_q = self.vq.codebook(tokens)              # (B, H, W, D)
        z_q = z_q.permute(0, 3, 1, 2)               # (B, D, H, W)
        return self.decode(z_q)

    def forward(self, x):
        """Full forward: encode → quantize → decode."""
        tokens, z_q, vq_loss = self.encode(x)
        recon = self.decode(z_q)
        return recon, tokens, vq_loss


tokenizer = Tokenizer(embed_dim=256, codebook_size=256).to(device)
n_params = sum(p.numel() for p in tokenizer.parameters())
print(f"Tokenizer: {n_params/1e6:.1f}M parameters")
print(f"Each 64×64 image → {4*4} = 16 discrete tokens from vocabulary of {256}")

In [ ]:
# ── Train the Tokenizer ──────────────────────────────────────────────

tok_optimizer = torch.optim.Adam(tokenizer.parameters(), lr=1e-4)
TOK_STEPS = 3000
TOK_BATCH = 64

tok_losses = {'recon': [], 'vq': []}
tokenizer.train()

pbar = tqdm(range(TOK_STEPS), desc='Training Tokenizer')
for step in pbar:
    imgs = buffer.sample_frames(TOK_BATCH).to(device)  # (B, 3, 64, 64)

    recon, tokens, vq_loss = tokenizer(imgs)
    recon_loss = F.mse_loss(recon, imgs)
    loss = recon_loss + vq_loss

    tok_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(tokenizer.parameters(), 10.0)
    tok_optimizer.step()

    tok_losses['recon'].append(recon_loss.item())
    tok_losses['vq'].append(vq_loss.item())

    if step % 500 == 0:
        pbar.set_postfix(recon=f"{recon_loss.item():.4f}", vq=f"{vq_loss.item():.4f}")

print(f"\nFinal — Recon: {np.mean(tok_losses['recon'][-100:]):.4f}, "
      f"VQ: {np.mean(tok_losses['vq'][-100:]):.4f}")

In [ ]:
# Visualize tokenizer reconstructions
tokenizer.eval()
test_imgs = buffer.sample_frames(8).to(device)

with torch.no_grad():
    recon, tokens, _ = tokenizer(test_imgs)

fig, axes = plt.subplots(3, 8, figsize=(16, 6))
for i in range(8):
    axes[0, i].imshow(test_imgs[i].permute(1, 2, 0).cpu())
    axes[0, i].set_title('Original', fontsize=8); axes[0, i].axis('off')

    axes[1, i].imshow(recon[i].permute(1, 2, 0).cpu().clamp(0, 1))
    axes[1, i].set_title('Reconstructed', fontsize=8); axes[1, i].axis('off')

    # Show the 4x4 token grid
    axes[2, i].imshow(tokens[i].cpu(), cmap='tab20', vmin=0, vmax=255)
    axes[2, i].set_title(f'Tokens (4×4)', fontsize=8); axes[2, i].axis('off')

fig.suptitle('VQ-VAE Tokenizer: 64×64 image → 16 tokens → reconstruction', fontsize=12)
plt.tight_layout(); plt.show()

# Codebook usage
all_tokens = []
for _ in range(50):
    imgs = buffer.sample_frames(64).to(device)
    with torch.no_grad():
        _, toks, _ = tokenizer(imgs)
    all_tokens.append(toks.cpu().flatten())
all_tokens = torch.cat(all_tokens)
unique = len(torch.unique(all_tokens))
print(f"Codebook utilization: {unique}/{tokenizer.codebook_size} entries used ({unique/tokenizer.codebook_size:.0%})")

---
## 4. Component 2: World Model (Autoregressive Transformer)

Now the key insight: we treat the sequence of tokens + actions as a **language** and train a GPT-like Transformer to predict the next token.

### Token Sequence Format

Each timestep is a **block** of 17 tokens: 16 observation tokens + 1 action token.

```
Step 0:  [obs₀ obs₁ obs₂ ... obs₁₅ act₀]
Step 1:  [obs₀ obs₁ obs₂ ... obs₁₅ act₁]
  ...        (total context: T × 17 tokens)
```

### What does the Transformer predict?

At each position, the Transformer predicts the **next token**:
- `obs₀` → predicts `obs₁` (next observation token)
- `obs₁₄` → predicts `obs₁₅` (last observation token)
- `obs₁₅` → this position is NOT used for obs prediction
- `act₀` → predicts `obs₀` of the NEXT timestep

Additionally, at the **action token position**, two extra heads predict:
- **Reward:** scalar regression (continuous reward value)
- **Done:** 2-class classification {continue, end}

### Attention: Causal (like GPT)

Standard causal (left-to-right) attention — each token can only attend to itself and all previous tokens. This is identical to GPT!

In [ ]:
class TransformerBlock(nn.Module):
    """GPT-2 style transformer block (pre-norm)."""

    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, attn_mask=None, kv_cache=None):
        # Self-attention with optional KV cache
        h = self.ln1(x)
        if kv_cache is not None:
            k_cache, v_cache = kv_cache
            k_new = h
            attn_out, _ = self.attn(h, torch.cat([k_cache, h], dim=1),
                                   torch.cat([v_cache, h], dim=1),
                                   attn_mask=None, is_causal=False)
            new_cache = (torch.cat([k_cache, h], dim=1), torch.cat([v_cache, h], dim=1))
        else:
            attn_out, _ = self.attn(h, h, h, attn_mask=attn_mask, is_causal=False)
            new_cache = None

        x = x + attn_out
        x = x + self.mlp(self.ln2(x))
        return x, new_cache


class WorldModelTransformer(nn.Module):
    """
    GPT-like world model: predicts next observation tokens, reward, and done
    from a causal sequence of [obs_tokens, action] blocks.
    """

    def __init__(self, codebook_size=256, num_actions=3, embed_dim=256,
                 num_layers=6, num_heads=4, tokens_per_block=17,
                 max_blocks=10, dropout=0.1):
        super().__init__()
        self.codebook_size = codebook_size
        self.num_actions = num_actions
        self.embed_dim = embed_dim
        self.tpb = tokens_per_block  # 17 = 16 obs + 1 act
        self.max_blocks = max_blocks
        self.max_tokens = tokens_per_block * max_blocks

        # Separate embeddings for obs tokens and action tokens
        self.obs_embed = nn.Embedding(codebook_size, embed_dim)
        self.act_embed = nn.Embedding(num_actions, embed_dim)
        self.pos_embed = nn.Embedding(self.max_tokens, embed_dim)

        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, dropout)
            for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(embed_dim)

        # Prediction heads
        self.obs_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.ReLU(), nn.Linear(embed_dim, codebook_size)
        )
        self.reward_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.ReLU(), nn.Linear(embed_dim, 1)  # scalar regression
        )
        self.done_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.ReLU(), nn.Linear(embed_dim, 2)  # {no, yes}
        )

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            m.weight.data.normal_(0, 0.02)
            if hasattr(m, 'bias') and m.bias is not None:
                m.bias.data.zero_()

    def _embed_sequence(self, obs_tokens, actions):
        """
        Interleave obs and action embeddings into the full sequence.
        obs_tokens: (B, T, 16) long — observation token indices
        actions:    (B, T) long — action indices
        Returns: (B, T*17) embedded sequence
        """
        B, T, K = obs_tokens.shape  # K=16

        # Embed observation tokens and actions
        obs_emb = self.obs_embed(obs_tokens)              # (B, T, 16, D)
        act_emb = self.act_embed(actions).unsqueeze(2)    # (B, T, 1, D)

        # Interleave: [obs₀..obs₁₅, act] per timestep
        block_emb = torch.cat([obs_emb, act_emb], dim=2)  # (B, T, 17, D)
        seq_emb = block_emb.reshape(B, T * self.tpb, self.embed_dim)  # (B, T*17, D)

        # Add positional embeddings
        S = seq_emb.shape[1]
        pos = self.pos_embed(torch.arange(S, device=seq_emb.device))
        return seq_emb + pos

    def forward(self, obs_tokens, actions):
        """
        Full forward pass for training.
        Returns obs logits, reward predictions, done logits at appropriate positions.
        """
        B, T, K = obs_tokens.shape
        S = T * self.tpb

        x = self._embed_sequence(obs_tokens, actions)  # (B, S, D)

        # Causal attention mask (lower triangular)
        mask = torch.triu(torch.ones(S, S, device=x.device), diagonal=1).bool()
        mask = mask.float().masked_fill(mask, float('-inf'))

        for block in self.blocks:
            x, _ = block(x, attn_mask=mask)
        x = self.ln_f(x)  # (B, S, D)

        # ── Extract predictions at the right positions ──

        # Obs prediction: from positions that predict the next obs token
        obs_logits = self.obs_head(x)  # (B, S, codebook_size)

        # Reward & done: only at action token positions (every 17th token, offset 16)
        act_positions = torch.arange(self.tpb - 1, S, self.tpb)  # [16, 33, 50, ...]
        act_hidden = x[:, act_positions]  # (B, T, D)
        reward_preds = self.reward_head(act_hidden)   # (B, T, 1) — scalar regression
        done_logits = self.done_head(act_hidden)      # (B, T, 2)

        return obs_logits, reward_preds, done_logits


world_model = WorldModelTransformer(
    codebook_size=256, num_actions=3, embed_dim=256,
    num_layers=6, num_heads=4, tokens_per_block=17,
    max_blocks=10, dropout=0.1
).to(device)

n_params = sum(p.numel() for p in world_model.parameters())
print(f"World Model: {n_params/1e6:.1f}M parameters")
print(f"Context: {world_model.max_blocks} steps × {world_model.tpb} tokens/step = {world_model.max_tokens} tokens")
print(f"Reward head: scalar regression (for dense continuous rewards)")
print(f"This is exactly like GPT, but instead of English words, it predicts visual tokens!")

In [ ]:
# ── World Model Loss ─────────────────────────────────────────────────

# Done class weights: most steps have done=False, so we boost done=True
done_weights = torch.tensor([1.0, 10.0], device=device)

def compute_world_model_loss(world_model, tokenizer, obs, actions, rewards, dones):
    """
    obs:     (B, T, 3, 64, 64) float [0,1]
    actions: (B, T) long
    rewards: (B, T) float — continuous dense rewards
    dones:   (B, T) long in {0, 1}
    """
    B, T = actions.shape

    # Tokenize all frames with the frozen tokenizer
    with torch.no_grad():
        obs_flat = obs.reshape(B * T, 3, 64, 64)
        tokens_flat, _, _ = tokenizer.encode(obs_flat)
        obs_tokens = tokens_flat.reshape(B, T, 4, 4)  # (B, T, 4, 4)
        obs_tokens = obs_tokens.reshape(B, T, 16)     # (B, T, 16)

    # Forward through world model
    obs_logits, reward_preds, done_logits = world_model(obs_tokens, actions)

    # ── Observation loss (cross-entropy, shifted by 1) ──
    S = T * world_model.tpb

    # Build the full target sequence (shifted left by 1)
    obs_labels = torch.full((B, S), -100, dtype=torch.long, device=obs.device)

    for t in range(T):
        block_start = t * world_model.tpb
        for k in range(15):
            obs_labels[:, block_start + k] = obs_tokens[:, t, k + 1]
        if t < T - 1:
            obs_labels[:, block_start + 16] = obs_tokens[:, t + 1, 0]

    loss_obs = F.cross_entropy(obs_logits.reshape(-1, world_model.codebook_size),
                               obs_labels.reshape(-1), ignore_index=-100)

    # ── Reward loss (MSE regression — dense rewards are continuous) ──
    reward_preds_flat = reward_preds.squeeze(-1)  # (B, T)
    loss_reward = 10.0 * F.mse_loss(reward_preds_flat, rewards)  # scaled up to match other losses

    # ── Done loss (weighted classification — boost rare done=True) ──
    loss_done = F.cross_entropy(done_logits.reshape(-1, 2), dones.reshape(-1),
                                weight=done_weights)

    total = loss_obs + loss_reward + loss_done
    return total, loss_obs.item(), loss_reward.item(), loss_done.item()

print("World model loss defined.")
print("Reward: MSE regression (10x scaled) — learns to predict continuous dense rewards.")
print("Done: weighted cross-entropy (10x boost for done=True).")

In [ ]:
# ── Train the World Model ────────────────────────────────────────────

tokenizer.eval()  # Freeze tokenizer during world model training

wm_optimizer = torch.optim.AdamW(world_model.parameters(), lr=1e-4, weight_decay=0.01)
WM_STEPS = 3000
WM_BATCH = 32
SEQ_LEN = 10  # 10 timesteps per sequence

wm_losses = {'obs': [], 'reward': [], 'done': []}
world_model.train()

pbar = tqdm(range(WM_STEPS), desc='Training World Model')
for step in pbar:
    obs, act, rew, done = buffer.sample_sequences(WM_BATCH, SEQ_LEN)
    obs, act, rew, done = obs.to(device), act.to(device), rew.to(device), done.to(device)

    total, lo, lr_, ld = compute_world_model_loss(world_model, tokenizer, obs, act, rew, done)

    wm_optimizer.zero_grad()
    total.backward()
    torch.nn.utils.clip_grad_norm_(world_model.parameters(), 10.0)
    wm_optimizer.step()

    wm_losses['obs'].append(lo)
    wm_losses['reward'].append(lr_)
    wm_losses['done'].append(ld)

    if step % 500 == 0:
        pbar.set_postfix(obs=f"{lo:.3f}", rew=f"{lr_:.3f}", done=f"{ld:.3f}")

print(f"\nFinal — Obs: {np.mean(wm_losses['obs'][-100:]):.3f}, "
      f"Reward: {np.mean(wm_losses['reward'][-100:]):.3f}, "
      f"Done: {np.mean(wm_losses['done'][-100:]):.3f}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
window = 50
for ax, key, title, color in [
    (axes[0], 'obs', 'Next-Token Prediction (CE)', '#5B8CB8'),
    (axes[1], 'reward', 'Reward Regression (10×MSE)', '#D97757'),
    (axes[2], 'done', 'Done Classification (CE)', '#7DA488')]:
    vals = wm_losses[key]
    smoothed = [np.mean(vals[max(0,i-window):i+1]) for i in range(len(vals))]
    ax.plot(smoothed, color=color, lw=1.5)
    ax.set_title(title); ax.set_xlabel('Step'); ax.grid(alpha=0.3)
fig.suptitle('World Model Training (GPT over visual tokens)', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Diagnostic: Does the world model predict rewards and dones correctly? ──
# This is critical — if the WM always predicts reward≈0 / done=False,
# the actor-critic will never see a reward signal in dreams.

world_model.eval(); tokenizer.eval()
all_rew_preds, all_rew_true = [], []
all_done_preds, all_done_true = [], []

for _ in range(100):
    obs, act, rew, done = buffer.sample_sequences(32, SEQ_LEN)
    obs, act, rew, done = obs.to(device), act.to(device), rew.to(device), done.to(device)
    with torch.no_grad():
        obs_flat = obs.reshape(-1, 3, 64, 64)
        toks, _, _ = tokenizer.encode(obs_flat)
        obs_tokens = toks.reshape(obs.shape[0], obs.shape[1], 16)
        _, rew_preds, done_logits = world_model(obs_tokens, act)
    all_rew_preds.append(rew_preds.squeeze(-1).cpu().flatten())
    all_rew_true.append(rew.cpu().flatten())
    all_done_preds.append(done_logits.argmax(-1).cpu().flatten())
    all_done_true.append(done.cpu().flatten())

rew_preds = torch.cat(all_rew_preds)
rew_true = torch.cat(all_rew_true)
done_preds = torch.cat(all_done_preds)
done_true = torch.cat(all_done_true)

print("=== World Model Reward/Done Diagnostics ===")
print(f"\nReward prediction (regression):")
print(f"  Predicted — mean: {rew_preds.mean():.4f}, std: {rew_preds.std():.4f}, "
      f"range: [{rew_preds.min():.3f}, {rew_preds.max():.3f}]")
print(f"  True      — mean: {rew_true.mean():.4f}, std: {rew_true.std():.4f}, "
      f"range: [{rew_true.min():.3f}, {rew_true.max():.3f}]")
mse = F.mse_loss(rew_preds, rew_true).item()
corr = torch.corrcoef(torch.stack([rew_preds, rew_true]))[0, 1].item()
print(f"  MSE: {mse:.4f}")
print(f"  Correlation: {corr:.3f}")

# Check terminal reward predictions
terminal_mask = (rew_true.abs() > 0.5)  # terminal rewards are ±1
if terminal_mask.sum() > 0:
    term_pred = rew_preds[terminal_mask]
    term_true = rew_true[terminal_mask]
    print(f"\n  Terminal rewards ({terminal_mask.sum().item()} samples):")
    print(f"    Predicted mean: {term_pred.mean():.3f} (should match true)")
    print(f"    True mean: {term_true.mean():.3f}")
    catch_mask = (rew_true > 0.5)
    miss_mask = (rew_true < -0.5)
    if catch_mask.sum() > 0:
        print(f"    Catch (+1) — predicted: {rew_preds[catch_mask].mean():.3f}")
    if miss_mask.sum() > 0:
        print(f"    Miss  (-1) — predicted: {rew_preds[miss_mask].mean():.3f}")

print(f"\nDone prediction (classification):")
print(f"  Predictions: {done_preds.bincount(minlength=2).tolist()}")
print(f"  Ground truth: {done_true.bincount(minlength=2).tolist()}")
print(f"  Accuracy: {(done_preds == done_true).float().mean():.1%}")
done_true_count = (done_true == 1).sum().item()
done_pred_correct = ((done_preds == 1) & (done_true == 1)).sum().item()
print(f"  Done=True recall: {done_pred_correct}/{done_true_count}")

if corr < 0.3:
    print("\n⚠️ WARNING: Low reward correlation — world model may not have learned reward structure.")
    print("Consider training longer or increasing reward loss weight.")

---
## 5. Imagined Rollouts (World Model Dreams)

Now let's see the world model **dream**. Given a starting observation + actions, it autoregressively generates future frames — exactly like GPT generating text, but generating images token by token.

```
1. Encode starting image → 16 tokens
2. Append action token
3. Transformer predicts next 16 obs tokens (one at a time)
4. Decode tokens → next frame image
5. Repeat from step 2
```

In [ ]:
@torch.no_grad()
def imagine_rollout(world_model, tokenizer, start_obs, actions, temperature=0.8):
    """
    Dream a trajectory inside the world model.

    start_obs: (1, 3, 64, 64) float [0,1] — starting frame
    actions: list of int — actions to take
    temperature: sampling temperature (lower = more deterministic)

    Returns: list of decoded images, rewards, dones
    """
    world_model.eval(); tokenizer.eval()

    # Encode starting frame
    tokens, _, _ = tokenizer.encode(start_obs.to(device))  # (1, 4, 4)
    tokens = tokens.reshape(1, 16)  # (1, 16)

    all_obs_tokens = [tokens]  # history of observation tokens
    all_actions = []
    pred_images = [tokenizer.decode_tokens(tokens.reshape(1, 4, 4))[0].cpu()]
    pred_rewards = []
    pred_dones = []

    for action in actions:
        all_actions.append(action)
        T = len(all_obs_tokens)

        # Build the full token sequence so far
        obs_tok_tensor = torch.stack(all_obs_tokens, dim=1)  # (1, T, 16)
        act_tensor = torch.tensor([all_actions], device=device)  # (1, T)

        # Forward through world model
        obs_logits, rew_preds, done_logits = world_model(obs_tok_tensor, act_tensor)

        # Get reward (scalar regression) and done from the last action position
        reward = rew_preds[0, -1, 0].item()  # scalar regression output
        done = torch.distributions.Categorical(logits=done_logits[0, -1]).sample().item()
        pred_rewards.append(reward)
        pred_dones.append(done)

        if done:
            break

        # Autoregressively generate next 16 observation tokens
        S = T * world_model.tpb
        next_tokens = []

        # First token: predicted from the action position
        logits = obs_logits[0, S - 1] / temperature
        t0 = torch.distributions.Categorical(logits=logits).sample()
        next_tokens.append(t0)

        # Remaining 15 tokens: feed each back through the model
        for k in range(1, 16):
            # Build sequence with partial next obs
            partial_next = torch.stack(next_tokens).unsqueeze(0)  # (1, k)
            full_next = torch.zeros(1, 16, dtype=torch.long, device=device)
            full_next[0, :k] = partial_next[0]

            # Full sequence: all previous blocks + partial new block
            new_obs = torch.cat([obs_tok_tensor, full_next.unsqueeze(1)], dim=1)
            new_act = torch.cat([act_tensor, torch.tensor([[0]], device=device)], dim=1)

            obs_l, _, _ = world_model(new_obs, new_act)
            # Position T*17 + (k-1) has token k-1, predicts token k
            pos = T * world_model.tpb + (k - 1)
            logits = obs_l[0, pos] / temperature
            tk = torch.distributions.Categorical(logits=logits).sample()
            next_tokens.append(tk)

        next_tokens = torch.stack(next_tokens).unsqueeze(0)  # (1, 16)
        all_obs_tokens.append(next_tokens)

        # Decode to image
        img = tokenizer.decode_tokens(next_tokens.reshape(1, 4, 4))
        pred_images.append(img[0].cpu())

    return pred_images, pred_rewards, pred_dones

print("Imagination function ready.")
print("This generates frames token-by-token, exactly like GPT generating words.")

In [ ]:
# Compare real vs imagined trajectories
env = CatchEnv()
n_compare = 3

fig, axes = plt.subplots(n_compare * 2, 10, figsize=(20, 3.5 * n_compare))

for trial in range(n_compare):
    # Play a real episode
    obs = env.reset()
    real_frames = [obs]
    ep_actions = []
    for _ in range(9):
        a = np.random.randint(3)
        obs, r, d = env.step(a)
        real_frames.append(obs)
        ep_actions.append(a)
        if d: break

    # Imagine with same starting frame and same actions
    start = torch.from_numpy(real_frames[0]).permute(2, 0, 1).float().unsqueeze(0) / 255.0
    dream_frames, dream_rewards, dream_dones = imagine_rollout(
        world_model, tokenizer, start, ep_actions, temperature=0.5)

    # Plot
    n_frames = min(10, len(real_frames), len(dream_frames))
    for i in range(n_frames):
        axes[trial*2, i].imshow(real_frames[i])
        axes[trial*2, i].set_title(f'Real t={i}', fontsize=7, color='#7DA488')
        axes[trial*2, i].axis('off')

        axes[trial*2+1, i].imshow(dream_frames[i].permute(1,2,0).clamp(0,1))
        axes[trial*2+1, i].set_title(f'Dream t={i}', fontsize=7, color='#D97757')
        axes[trial*2+1, i].axis('off')

    for i in range(n_frames, 10):
        axes[trial*2, i].axis('off')
        axes[trial*2+1, i].axis('off')

fig.suptitle('Real Environment vs World Model Dreams (same start + actions)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 6. Component 3: Actor-Critic (Policy Trained in Dreams)

Now the final piece: we train a policy **entirely inside the world model's imagination**.

The actor-critic:
1. Sees **reconstructed images** from the tokenizer (never raw pixels)
2. Uses a CNN to extract features, then an LSTM for temporal context
3. Outputs action probabilities (actor) and a value estimate (critic)

Training loop:
1. Sample a starting frame from the buffer
2. Let the actor-critic choose actions
3. Feed those actions to the world model to imagine the next frame + reward
4. Compute advantage and update the actor-critic with A2C

**The actor-critic never sees the real environment during training!**

In [ ]:
class ActorCritic(nn.Module):
    """
    CNN + LSTM actor-critic.
    Input: reconstructed 64x64 RGB image from the tokenizer.
    """

    def __init__(self, num_actions=3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=1, padding=1), nn.MaxPool2d(2), nn.ReLU(),  # 64→32
            nn.Conv2d(32, 32, 3, stride=1, padding=1), nn.MaxPool2d(2), nn.ReLU(), # 32→16
            nn.Conv2d(32, 64, 3, stride=1, padding=1), nn.MaxPool2d(2), nn.ReLU(), # 16→8
            nn.Conv2d(64, 64, 3, stride=1, padding=1), nn.MaxPool2d(2), nn.ReLU(), # 8→4
        )  # Output: (B, 64, 4, 4) = 1024

        self.lstm = nn.LSTMCell(1024, 512)
        self.actor = nn.Linear(512, num_actions)
        self.critic = nn.Linear(512, 1)

    def forward(self, obs, hx=None):
        """
        obs: (B, 3, 64, 64) float [0, 1]
        hx: (h, c) LSTM hidden state, or None
        Returns: action_logits, value, new_hx
        """
        B = obs.shape[0]
        feat = self.cnn(obs).reshape(B, -1)  # (B, 1024)

        if hx is None:
            hx = (torch.zeros(B, 512, device=obs.device),
                  torch.zeros(B, 512, device=obs.device))

        h, c = self.lstm(feat, hx)
        logits = self.actor(h)      # (B, num_actions)
        value = self.critic(h)[:, 0] # (B,)
        return logits, value, (h, c)


actor_critic = ActorCritic(num_actions=3).to(device)
n_params = sum(p.numel() for p in actor_critic.parameters())
print(f"Actor-Critic: {n_params/1e6:.1f}M parameters")
print(f"This policy will be trained ENTIRELY in dreams — never seeing the real environment.")

In [ ]:
# ── Dream Training Environment ──────────────────────────────────────

class DreamEnv:
    """
    Wraps the world model as a step-able environment for the actor-critic.

    Three critical design choices:
    1. reset() samples from EPISODE STARTS (not random mid-episode frames)
       — so dreams always begin with ball at top, like real episodes
    2. step() uses a 2-pass token generation scheme:
       Pass 1: get reward, done, and first obs token from action position
       Pass 2: fill current-frame tokens as initialization, predict tokens 1-15
       — this is efficient (2 forward passes) while being accurate
    3. Reward is read as a scalar (regression), matching dense reward training
    """

    def __init__(self, world_model, tokenizer, buffer, device):
        self.wm = world_model
        self.tok = tokenizer
        self.buffer = buffer
        self.device = device

    @torch.no_grad()
    def reset(self, batch_size=32):
        """Sample starting frames from episode BEGINNINGS."""
        self.batch_size = batch_size

        # Pick the first frame (reset frame) from random episodes
        # This ensures dreams start with ball at top, like real episodes
        frames = []
        for _ in range(batch_size):
            ep = self.buffer.episodes[np.random.randint(len(self.buffer.episodes))]
            frames.append(ep['observations'][0])  # first frame = reset frame
        imgs = torch.from_numpy(np.stack(frames)).permute(0, 3, 1, 2).float().to(self.device) / 255.0

        tokens, _, _ = self.tok.encode(imgs)
        self.obs_tokens = tokens.reshape(batch_size, 1, 16)  # (B, 1, 16)
        self.all_actions = []  # no actions yet

        # Return reconstructed image (actor-critic sees what tokenizer produces)
        recon = self.tok.decode_tokens(tokens)
        return recon.clamp(0, 1)

    @torch.no_grad()
    def step(self, actions):
        """
        actions: (B,) long tensor
        Returns: next_obs (B,3,64,64), rewards (B,), dones (B,)

        Uses 2-pass token generation:
          Pass 1: full forward → token 0, reward, done
          Pass 2: init new block with current frame's tokens → tokens 1-15
        """
        B = self.batch_size
        self.all_actions.append(actions)
        T = self.obs_tokens.shape[1]

        # Keep only last few steps to fit in context
        max_hist = min(T, self.wm.max_blocks - 1)
        obs_hist = self.obs_tokens[:, -max_hist:]  # (B, H, 16)
        act_hist = torch.stack(self.all_actions[-max_hist:], dim=1)  # (B, H)

        # ── Pass 1: get reward, done, and first predicted token ──
        obs_logits, rew_preds, done_logits = self.wm(obs_hist, act_hist)

        # Reward (scalar regression) and done from last action position
        rewards = rew_preds[:, -1, 0]  # (B,) scalar
        dones = done_logits[:, -1].argmax(-1).bool()  # (B,)

        # Token 0: predicted from the action position (last position in sequence)
        S = max_hist * self.wm.tpb
        t0 = torch.distributions.Categorical(logits=obs_logits[:, S - 1] / 0.8).sample()

        # ── Pass 2: predict tokens 1-15 ──
        # Initialize new block with CURRENT frame's tokens (most don't change between frames)
        # then overwrite token 0 with our prediction
        current_tokens = obs_hist[:, -1]  # (B, 16) — last observation
        next_block = current_tokens.clone()
        next_block[:, 0] = t0

        # Append the initialized new block and run forward
        new_obs = torch.cat([obs_hist, next_block.unsqueeze(1)], dim=1)  # (B, H+1, 16)
        dummy_act = torch.zeros(B, 1, dtype=torch.long, device=self.device)
        new_act = torch.cat([act_hist, dummy_act], dim=1)  # (B, H+1)

        obs_l2, _, _ = self.wm(new_obs, new_act)

        # Read predicted tokens 1-15 from the new block's positions
        next_tokens = [t0]
        for k in range(1, 16):
            # Position (H*17 + k-1) has token k-1, predicts token k
            pos = max_hist * self.wm.tpb + (k - 1)
            tk = torch.distributions.Categorical(logits=obs_l2[:, pos] / 0.8).sample()
            next_tokens.append(tk)

        next_tokens = torch.stack(next_tokens, dim=1)  # (B, 16)

        # Append to history
        self.obs_tokens = torch.cat([self.obs_tokens, next_tokens.unsqueeze(1)], dim=1)

        # Decode to image
        next_img = self.tok.decode_tokens(next_tokens.reshape(B, 4, 4))

        return next_img.clamp(0, 1), rewards, dones

print("Dream environment ready — 3 fixes applied:")
print("  1. reset() samples from episode starts (ball at top)")
print("  2. step() uses 2-pass token generation (correct positions)")
print("  3. Reward read as scalar regression (not 3-class categorical)")

In [ ]:
# ── Advantage Estimation (GAE-lambda) ────────────────────────────────

def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    """
    Generalized Advantage Estimation.
    rewards, values, dones: lists of (B,) tensors, length T.
    Returns: advantages (T, B), returns (T, B)
    """
    T = len(rewards)
    advantages = [torch.zeros_like(rewards[0])] * T
    last_gae = torch.zeros_like(rewards[0])

    for t in reversed(range(T)):
        if t == T - 1:
            next_val = torch.zeros_like(values[0])
        else:
            next_val = values[t + 1]
        mask = 1.0 - dones[t].float()
        delta = rewards[t] + gamma * next_val * mask - values[t]
        last_gae = delta + gamma * lam * mask * last_gae
        advantages[t] = last_gae

    advantages = torch.stack(advantages)  # (T, B)
    returns = advantages + torch.stack(values)  # (T, B)
    return advantages, returns

print("GAE-lambda advantage estimation defined.")

In [ ]:
# ── Train Actor-Critic in Dreams ─────────────────────────────────────

world_model.eval(); tokenizer.eval()  # Freeze both during actor-critic training
dream_env = DreamEnv(world_model, tokenizer, buffer, device)
ac_optimizer = torch.optim.Adam(actor_critic.parameters(), lr=3e-4)

AC_EPOCHS = 300       # more epochs since sparse reward is harder
DREAM_BATCH = 64
HORIZON = 15          # match actual episode length (~14 steps)
ENTROPY_COEF = 0.01
VALUE_COEF = 0.5

ac_losses = {'policy': [], 'value': [], 'entropy': [], 'mean_reward': []}

pbar = tqdm(range(AC_EPOCHS), desc='Dream Training')
for epoch in pbar:
    actor_critic.train()

    # Reset dream environment
    obs = dream_env.reset(DREAM_BATCH)
    hx = None

    log_probs_list, values_list, rewards_list, dones_list, entropies_list = [], [], [], [], []

    for t in range(HORIZON):
        logits, value, hx = actor_critic(obs, hx)
        dist = torch.distributions.Categorical(logits=logits)
        action = dist.sample()

        log_probs_list.append(dist.log_prob(action))
        values_list.append(value)
        entropies_list.append(dist.entropy())

        obs, rewards, dones = dream_env.step(action)
        rewards_list.append(rewards)
        dones_list.append(dones)

        # Detach LSTM state to prevent backprop through world model
        hx = (hx[0].detach(), hx[1].detach())

    # Compute GAE
    advantages, returns = compute_gae(rewards_list, values_list, dones_list)

    # Policy loss (REINFORCE with baseline)
    log_probs = torch.stack(log_probs_list)  # (T, B)
    policy_loss = -(log_probs * advantages.detach()).mean()

    # Value loss
    values = torch.stack(values_list)  # (T, B)
    value_loss = F.mse_loss(values, returns.detach())

    # Entropy bonus (encourage exploration)
    entropy = torch.stack(entropies_list).mean()

    total_loss = policy_loss + VALUE_COEF * value_loss - ENTROPY_COEF * entropy

    ac_optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(actor_critic.parameters(), 10.0)
    ac_optimizer.step()

    mean_rew = torch.stack(rewards_list).sum(0).mean().item()
    ac_losses['policy'].append(policy_loss.item())
    ac_losses['value'].append(value_loss.item())
    ac_losses['entropy'].append(entropy.item())
    ac_losses['mean_reward'].append(mean_rew)

    if epoch % 50 == 0:
        pbar.set_postfix(rew=f"{mean_rew:.2f}", pi=f"{policy_loss.item():.3f}")

print(f"\nFinal dream reward: {np.mean(ac_losses['mean_reward'][-20:]):.2f}")

In [ ]:
# Plot actor-critic training
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(ac_losses['mean_reward'], color='#D97757', lw=1.5, alpha=0.5)
# Smoothed
w = 20
smoothed = [np.mean(ac_losses['mean_reward'][max(0,i-w):i+1]) for i in range(len(ac_losses['mean_reward']))]
axes[0].plot(smoothed, color='#D97757', lw=2)
axes[0].set_title('Dream Reward (total per episode)'); axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=0.3)
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

axes[1].plot(ac_losses['policy'], color='#5B8CB8', lw=1); axes[1].set_title('Policy Loss')
axes[1].set_xlabel('Epoch'); axes[1].grid(alpha=0.3)

axes[2].plot(ac_losses['entropy'], color='#7DA488', lw=1); axes[2].set_title('Entropy')
axes[2].set_xlabel('Epoch'); axes[2].grid(alpha=0.3)

fig.suptitle('Actor-Critic Training (entirely in world model dreams!)', fontweight='bold')
plt.tight_layout(); plt.show()

---
## 7. Evaluation: Dream-Trained Policy in the Real Environment

The ultimate test: does a policy trained **entirely in dreams** actually work in the **real environment**?

The actor-critic has never seen a real Catch game — it only experienced the world model's imagined frames. Let's see if it learned to catch the ball.

In [ ]:
@torch.no_grad()
def evaluate_policy(actor_critic, tokenizer, env, n_episodes=200):
    """Play the REAL environment with the dream-trained policy."""
    actor_critic.eval(); tokenizer.eval()
    catches, misses = 0, 0
    all_rewards = []

    for _ in range(n_episodes):
        obs = env.reset()
        hx = None
        total_reward = 0
        done = False

        while not done:
            # Convert observation to tensor
            obs_t = torch.from_numpy(obs).permute(2, 0, 1).float().unsqueeze(0).to(device) / 255.0

            # IMPORTANT: pass through tokenizer encode-decode
            # (actor-critic was trained on reconstructed images, not raw pixels)
            recon = tokenizer.decode_tokens(
                tokenizer.encode(obs_t)[0]).clamp(0, 1)

            logits, _, hx = actor_critic(recon, hx)
            action = logits.argmax(dim=-1).item()  # greedy

            obs, reward, done = env.step(action)
            total_reward += reward

        if total_reward > 0: catches += 1
        else: misses += 1
        all_rewards.append(total_reward)

    return catches, misses, np.mean(all_rewards)


# Evaluate!
env = CatchEnv()
catches, misses, avg_r = evaluate_policy(actor_critic, tokenizer, env, n_episodes=200)

print(f"\n{'='*50}")
print(f"  DREAM-TRAINED POLICY ON REAL ENVIRONMENT")
print(f"{'='*50}")
print(f"  Catches: {catches}/200 ({catches/200:.0%})")
print(f"  Misses:  {misses}/200")
print(f"  Avg reward: {avg_r:.2f}")
print(f"  Random baseline: ~33%")
print(f"{'='*50}")

In [ ]:
# Watch the policy play
env = CatchEnv()
n_show = 4
fig, axes = plt.subplots(n_show, 10, figsize=(20, 2.5 * n_show))

actor_critic.eval(); tokenizer.eval()
action_names = ['←', '·', '→']

for trial in range(n_show):
    obs = env.reset()
    hx = None
    frames = [obs.copy()]
    actions_taken = []

    for _ in range(12):
        obs_t = torch.from_numpy(obs).permute(2, 0, 1).float().unsqueeze(0).to(device) / 255.0
        with torch.no_grad():
            recon = tokenizer.decode_tokens(tokenizer.encode(obs_t)[0]).clamp(0, 1)
            logits, _, hx = actor_critic(recon, hx)
        action = logits.argmax(dim=-1).item()
        actions_taken.append(action)
        obs, reward, done = env.step(action)
        frames.append(obs.copy())
        if done: break

    result = '✓ CATCH' if reward > 0 else '✗ MISS'
    color = '#7DA488' if reward > 0 else '#D97757'

    for i in range(min(10, len(frames))):
        axes[trial, i].imshow(frames[i])
        lbl = action_names[actions_taken[i]] if i < len(actions_taken) else ''
        axes[trial, i].set_title(lbl, fontsize=10)
        axes[trial, i].axis('off')
    for i in range(len(frames), 10):
        axes[trial, i].axis('off')

    axes[trial, 0].set_ylabel(result, fontsize=11, color=color, fontweight='bold', rotation=0, labelpad=50)

fig.suptitle('Dream-Trained Policy Playing Real Catch (← = left, · = stay, → = right)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 8. Analysis: Understanding the Discrete Representation

Let's look at what the tokenizer actually learned — which visual patterns map to which codebook entries.

In [ ]:
# Analyze how tokens change as the ball moves
env = CatchEnv()
tokenizer.eval()

obs = env.reset()
token_maps = []
frame_imgs = []

for _ in range(15):
    obs_t = torch.from_numpy(obs).permute(2, 0, 1).float().unsqueeze(0).to(device) / 255.0
    with torch.no_grad():
        tokens, _, _ = tokenizer.encode(obs_t)
    token_maps.append(tokens[0].cpu().numpy())  # (4, 4)
    frame_imgs.append(obs.copy())
    obs, r, d = env.step(1)  # stay
    if d: break

n_frames = len(token_maps)
fig, axes = plt.subplots(2, n_frames, figsize=(2.2 * n_frames, 5))

for i in range(n_frames):
    axes[0, i].imshow(frame_imgs[i])
    axes[0, i].set_title(f't={i}', fontsize=8); axes[0, i].axis('off')

    im = axes[1, i].imshow(token_maps[i], cmap='tab20', vmin=0, vmax=20)
    # Annotate with token indices
    for y in range(4):
        for x in range(4):
            axes[1, i].text(x, y, str(token_maps[i][y, x]),
                           ha='center', va='center', fontsize=6, color='white',
                           fontweight='bold')
    axes[1, i].axis('off')

fig.suptitle('Token IDs change as the ball moves down\n'
             'Each 16×16 pixel region maps to one codebook entry', fontsize=11)
plt.tight_layout(); plt.show()

# Show which tokens changed between frames
print("\nToken changes between consecutive frames:")
for i in range(1, min(6, n_frames)):
    diff = (token_maps[i] != token_maps[i-1]).sum()
    print(f"  t={i-1}→{i}: {diff}/16 tokens changed")

In [ ]:
# World model's reward and done prediction quality
world_model.eval(); tokenizer.eval()

all_rew_preds, all_rew_true = [], []
correct_done, total = 0, 0

for _ in range(50):
    obs, act, rew, done = buffer.sample_sequences(32, SEQ_LEN)
    obs, act, rew, done = obs.to(device), act.to(device), rew.to(device), done.to(device)

    with torch.no_grad():
        obs_flat = obs.reshape(-1, 3, 64, 64)
        toks, _, _ = tokenizer.encode(obs_flat)
        obs_tokens = toks.reshape(obs.shape[0], obs.shape[1], 16)
        _, rew_preds, done_logits = world_model(obs_tokens, act)

    all_rew_preds.append(rew_preds.squeeze(-1).cpu().flatten())
    all_rew_true.append(rew.cpu().flatten())

    done_pred = done_logits.argmax(-1)
    correct_done += (done_pred == done).sum().item()
    total += done.numel()

rew_preds = torch.cat(all_rew_preds)
rew_true = torch.cat(all_rew_true)
mse = F.mse_loss(rew_preds, rew_true).item()
corr = torch.corrcoef(torch.stack([rew_preds, rew_true]))[0, 1].item()

# Check catch/miss prediction accuracy (sign agreement on terminal rewards)
terminal = rew_true.abs() > 0.5
if terminal.sum() > 0:
    sign_match = ((rew_preds[terminal] > 0) == (rew_true[terminal] > 0)).float().mean().item()
else:
    sign_match = float('nan')

print(f"World Model Prediction Quality:")
print(f"  Reward MSE:         {mse:.4f}")
print(f"  Reward correlation: {corr:.3f}")
print(f"  Catch/miss sign acc: {sign_match:.1%} (on terminal steps)")
print(f"  Done accuracy:      {correct_done/total:.1%}")

---
## Summary

We built **IRIS** from scratch — a world model that treats visual environments as a language modeling problem.

### The Three Components

| Component | What it does | Architecture | Analogy |
|-----------|-------------|-------------|----------|
| **Tokenizer** | Image → discrete tokens | VQ-VAE | BPE tokenizer in GPT |
| **World Model** | Predict next tokens + reward + done | Autoregressive Transformer | GPT itself |
| **Actor-Critic** | Choose actions, estimate value | CNN + LSTM | RL agent |

### Key Insights

1. **Discretization is the bridge.** VQ-VAE converts continuous pixels into discrete tokens, making visual prediction a classification problem (cross-entropy) rather than a regression problem (MSE). This is more stable and enables sampling.

2. **World modeling = language modeling.** Once images are tokens, predicting the next frame is literally next-token prediction — the same problem GPT solves for text.

3. **Dream training works.** The actor-critic never saw the real environment during training. It learned purely from the world model's imagined rollouts. This is incredibly sample-efficient.

4. **The key limitation:** Everything depends on tokenizer quality. If the tokenizer can't represent important visual details, the world model and policy can't learn them.

### vs. DINO-WM

| Aspect | DINO-WM | IRIS |
|--------|---------|------|
| Tokens | Continuous (384-dim DINO features) | Discrete (integers from codebook) |
| Encoder training | None (frozen DINO) | Trained VQ-VAE |
| Prediction loss | MSE (regression) | Cross-entropy (classification) |
| Planning | CEM in embedding space | Full RL policy in dreams |
| Strength | Zero-shot transfer via pre-trained features | Sample efficiency, stable training |

### Going Further
- **DIAMOND** replaces the autoregressive Transformer with a diffusion model
- **DreamerV3** uses continuous latent states (RSSM) instead of discrete tokens
- **Delta-IRIS** improves efficiency by encoding frame *differences*
- Scale to Atari with `pip install gymnasium[atari]` and the full IRIS codebase

### References
- [IRIS Paper](https://arxiv.org/abs/2209.00588) | [Code](https://github.com/eloialonso/iris)
- [VQ-VAE Paper](https://arxiv.org/abs/1711.00937)
- [Taming Transformers](https://arxiv.org/abs/2012.09841) (VQ-GAN architecture used in IRIS)